# Sprint 37 — Rapport de release GitLab

Ce notebook **audite** la transformation « dépôt de travail → version GitLab » produite par
`scripts/prepare_gitlab_release.py` (pipeline du Sprint 37), sans rien publier.

Il répond à trois questions :

1. **Quels fichiers** sont conservés, exclus, ou remplacés par des docs neutres ?
2. **Quelles traces d'outillage interne** existent dans le dépôt source, et où ?
3. **L'export sortirait-il propre** (0 trace) ?

> ⚠️ **Non destructif.** Le dépôt source n'est jamais modifié ni poussé. L'export est construit
> dans un **dossier temporaire jetable**, scanné, puis supprimé. Aucun commit, aucun push, aucune
> écriture dans le `output_dir` de la config (`../cl-embedded-gitlab`).

Les fonctions utilisées sont **importées directement** depuis les scripts du pipeline, ce qui garantit
que ce rapport reflète exactement le comportement réel de `make gitlab-release`.

In [1]:
import sys
import shutil
import subprocess
import tempfile
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# ── Résolution dynamique de la racine du dépôt ──
ROOT = Path.cwd()
while not (ROOT / "experiments").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# git ls-files (utilisé par build_plan/export) échappe les chemins non-ASCII par
# défaut → core.quotepath=false les rend en UTF-8 brut, copiables tels quels.
subprocess.run(["git", "-C", str(ROOT), "config", "core.quotepath", "false"], check=True)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))  # check_ai_traces / prepare_gitlab_release

from check_ai_traces import load_config, scan_tree, Finding  # noqa: E402
from prepare_gitlab_release import build_plan, export, apply_rewrites  # noqa: E402
from src.evaluation.plots import save_figure  # noqa: E402

FIGS = ROOT / "docs/figures/sprint37_gitlab_release"
FIGS.mkdir(parents=True, exist_ok=True)

config = load_config(ROOT / "configs/gitlab_release.yaml")
print(f"ROOT          : {ROOT}")
print(f"output_dir    : {config['output_dir']} (non touché par ce notebook)")
print(f"exclude_paths : {len(config['exclude_paths'])} règles")
print(f"patterns      : {len(config['forbidden_patterns'])} patterns interdits")

ROOT          : /home/leonard/Documents/ENAC/cl-embedded
output_dir    : ../cl-embedded-gitlab (non touché par ce notebook)
exclude_paths : 13 règles
patterns      : 9 patterns interdits


## 1. Plan de transformation

`build_plan` énumère les fichiers **suivis par git** (`git ls-files`, donc pas de data ni d'ignorés),
puis les partitionne en **conservés** vs **exclus**. Les **docs neutres** (README/CONTRIBUTING)
sont déposées par-dessus l'export.

In [2]:
plan = build_plan(ROOT, config)
n_kept = len(plan["kept"])
n_excluded = len(plan["excluded"])
n_neutral = len(plan["neutral_docs"])
print(f"Conservés   : {n_kept}")
print(f"Exclus      : {n_excluded}")
print(f"Docs neutres: {n_neutral}")

fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Conservés", "Exclus", "Docs neutres"]
values = [n_kept, n_excluded, n_neutral]
colors = ["#2e7d32", "#c62828", "#1565c0"]
bars = ax.bar(labels, values, color=colors)
ax.bar_label(bars, padding=3)
ax.set_ylabel("Nombre de fichiers")
ax.set_title("Vue d'ensemble de la release GitLab")
ax.margins(y=0.15)
save_figure(fig, FIGS / "release_overview.png")

Conservés   : 5303
Exclus      : 3049
Docs neutres: 2
[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint37_gitlab_release/release_overview.png


## 2. Répartition des fichiers conservés

Où se concentre le code qui partira sur GitLab ? Regroupement par répertoire de premier niveau.

In [3]:
def top_dir(rel: str) -> str:
    parts = Path(rel).parts
    return parts[0] if len(parts) > 1 else "(racine)"

by_dir = Counter(top_dir(r) for r in plan["kept"])
by_ext = Counter(Path(r).suffix or "(sans ext)" for r in plan["kept"])

df_dir = (pd.DataFrame(by_dir.items(), columns=["dossier", "fichiers"])
          .sort_values("fichiers", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(7, max(3, 0.4 * len(df_dir))))
ax.barh(df_dir["dossier"], df_dir["fichiers"], color="#2e7d32")
ax.invert_yaxis()
ax.set_xlabel("Fichiers conservés")
ax.set_title("Fichiers conservés par répertoire de 1er niveau")
for i, v in enumerate(df_dir["fichiers"]):
    ax.text(v + 0.5, i, str(v), va="center")
save_figure(fig, FIGS / "kept_by_directory.png")

print("Top extensions conservées :")
display(pd.DataFrame(by_ext.most_common(10), columns=["extension", "fichiers"]))
df_dir

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint37_gitlab_release/kept_by_directory.png
Top extensions conservées :


,extension,fichiers
0,.json,1376
1,.c,996
2,.png,695
3,.yaml,576
4,.md,395
5,.h,303
6,.npy,230
7,.py,188
8,.txt,143
9,.ipynb,119


,dossier,fichiers
0,experiments,2252
1,firmware,1574
2,notebooks,619
3,docs,451
4,configs,208
5,scripts,74
6,src,62
7,tests,51
8,(racine),10
9,.github,1


## 3. Détail des exclusions

Chaque fichier exclu est attribué à la règle `exclude_paths` qui l'a capturé (même logique de
préfixe que `is_excluded`).

In [4]:
def matching_rule(rel: str, exclude_paths: list[str]) -> str:
    for pre in exclude_paths:
        p = pre.rstrip("/")
        if rel == p or rel.startswith(p + "/"):
            return pre
    return "(aucune)"

excl_rows = [{"fichier": rel, "règle": matching_rule(rel, config["exclude_paths"])}
             for rel in plan["excluded"]]
df_excl = pd.DataFrame(excl_rows)
by_rule = (df_excl.groupby("règle").size()
           .sort_values(ascending=False) if not df_excl.empty else pd.Series(dtype=int))

if not by_rule.empty:
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(by_rule))))
    ax.barh(by_rule.index.astype(str), by_rule.values, color="#c62828")
    ax.invert_yaxis()
    ax.set_xlabel("Fichiers exclus")
    ax.set_title("Fichiers exclus par règle exclude_paths")
    for i, v in enumerate(by_rule.values):
        ax.text(v + 0.05, i, str(v), va="center")
    save_figure(fig, FIGS / "excluded_by_rule.png")
else:
    print("Aucun fichier exclu.")

df_excl

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint37_gitlab_release/excluded_by_rule.png


,fichier,règle
0,.github/workflows/ai-trace-guard.yml,.github/workflows/ai-trace-guard.yml
1,CLAUDE.md,CLAUDE.md
2,Makefile,Makefile
3,configs/gitlab_release.yaml,configs/gitlab_release.yaml
4,docs/gitlab/CONTRIBUTING.md,docs/gitlab/
...,...,...
3044,skills/model_implementation.md,skills/
3045,skills/model_improvement_v2.md,skills/
3046,skills/sprint_generation.md,skills/
3047,skills/update_pipeline_diagram.md,skills/


## 4. Scan de traces — dépôt source

- **Mode `--source`** : ignore les zones internes connues (CLAUDE.md, `skills/`, `graphify-out/`, …).
  Tout match ici serait une trace **non couverte** → doit être **0** (sinon ajouter un
  `exclude_path`/`rewrite_rule` dans `configs/gitlab_release.yaml`).
- **Mode brut** : compte toutes les traces, y compris dans les zones exclues — illustre *pourquoi*
  ces zones sont retirées.

In [5]:
findings_src = scan_tree(ROOT, config, source_mode=True)
findings_raw = scan_tree(ROOT, config, source_mode=False)
print(f"Traces hors zones connues (--source) : {len(findings_src)}  "
      f"{'✅ propre' if not findings_src else '❌ à couvrir'}")
print(f"Traces brutes (toutes zones)         : {len(findings_raw)}")

by_pattern = Counter(f.pattern for f in findings_raw)
df_pat = (pd.DataFrame(by_pattern.items(), columns=["pattern", "matches"])
          .sort_values("matches", ascending=False).reset_index(drop=True))

if not df_pat.empty:
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(df_pat))))
    ax.barh(df_pat["pattern"], df_pat["matches"], color="#ef6c00")
    ax.invert_yaxis()
    ax.set_xlabel("Occurrences (dépôt source, toutes zones)")
    ax.set_title("Traces d'outillage interne par pattern interdit")
    for i, v in enumerate(df_pat["matches"]):
        ax.text(v + 0.5, i, str(v), va="center")
    save_figure(fig, FIGS / "traces_by_pattern.png")
else:
    print("Aucune trace détectée dans le dépôt source.")

if findings_src:
    print("\n⚠️ Traces NON couvertes (à traiter dans gitlab_release.yaml) :")
    display(pd.DataFrame([f.__dict__ for f in findings_src]))
df_pat

Traces hors zones connues (--source) : 166  ❌ à couvrir
Traces brutes (toutes zones)         : 3216
[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint37_gitlab_release/traces_by_pattern.png

⚠️ Traces NON couvertes (à traiter dans gitlab_release.yaml) :


,path,line_no,pattern,excerpt
0,README.md,134,\bclaude\b,├── CLAUDE.md # Context for ...
1,cl_embedded.egg-info/PKG-INFO,104,\bclaude\b,├── CLAUDE.md # Context for ...
2,cl_embedded.egg-info/PKG-INFO,125,\bclaude\b,├── skills/ # Claude promp...
3,configs/board_pair_maha_ewc.yaml,4,\bclaude\b,# Aucun hyperparamètre en dur dans le code (rè...
4,configs/board_pair_maha_ewc.yaml,48,\bclaude\b,"# native, pour le désaccord vs Mahalanobis. Au..."
...,...,...,...,...
161,src/evaluation/autonomy.py,14,\bclaude\b,Règle CLAUDE.md :
162,src/evaluation/compute_cost.py,7,\bclaude\b,cf. CLAUDE.md).
163,src/evaluation/hw_cost_model.py,16,\bclaude\b,(règle CLAUDE.md). Ce module ne contient que d...
164,src/evaluation/streaming_model.py,18,\bclaude\b,`configs/streaming_profile.yaml` (règle CLAUDE...


,pattern,matches
0,<exclude_path résiduel>,3050
1,\bclaude\b,147
2,\bgraphify\b,15
3,\.claude/,2
4,\bclaude-[a-z0-9._-]+,2


## 5. Export jetable + gate dur

On reconstruit l'export complet dans un **dossier temporaire** (exclusions + réécritures + docs
neutres) puis on relance le scanner en **mode export** (`source_mode=False`). Le gate du pipeline
exige **0 trace** : c'est ce que l'on vérifie ici. Le dossier est supprimé ensuite.

In [6]:
tmp_dir = Path(tempfile.mkdtemp(prefix="gitlab-release-preview-"))
try:
    export(ROOT, tmp_dir, config)
    findings_export = scan_tree(tmp_dir, config, source_mode=False)
    n_export_files = sum(1 for p in tmp_dir.rglob("*") if p.is_file())
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"Fichiers dans l'export jetable : {n_export_files}")
if findings_export:
    print(f"❌ Gate ÉCHEC — {len(findings_export)} trace(s) résiduelle(s) :")
    display(pd.DataFrame([f.__dict__ for f in findings_export]))
else:
    print("✅ Gate OK — l'export sortirait propre : 0 trace.")
print(f"(dossier temporaire supprimé ; '{config['output_dir']}' jamais touché)")

Fichiers dans l'export jetable : 5304
❌ Gate ÉCHEC — 4 trace(s) résiduelle(s) :


,path,line_no,pattern,excerpt
0,notebooks/notebooks/04_final_comparison_execut...,330,\bclaude\b,"""/tmp/claude-1002/ipykernel_121955/1637571768...."
1,notebooks/notebooks/04_final_comparison_execut...,330,\bclaude-[a-z0-9._-]+,"""/tmp/claude-1002/ipykernel_121955/1637571768...."
2,notebooks/notebooks/04_final_comparison_execut...,408,\bclaude\b,"""/tmp/claude-1002/ipykernel_121955/299245058.p..."
3,notebooks/notebooks/04_final_comparison_execut...,408,\bclaude-[a-z0-9._-]+,"""/tmp/claude-1002/ipykernel_121955/299245058.p..."


(dossier temporaire supprimé ; '../cl-embedded-gitlab' jamais touché)


## 6. Avant / après d'une réécriture

Les fichiers conservés passent par `apply_rewrites` (substitutions, suppression de lignes/sections)
avant le scan. Illustration sur un markdown contenant des mentions neutralisées.

In [7]:
rules = config.get("rewrite_rules", [])

def first_changed_md() -> str | None:
    for rel in plan["kept"]:
        if not rel.endswith(".md"):
            continue
        try:
            src = (ROOT / rel).read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            continue
        if apply_rewrites(src, rel, rules) != src:
            return rel
    return None

sample = first_changed_md()
if sample is None:
    print("Aucun markdown conservé n'est modifié par les règles de réécriture.")
else:
    before = (ROOT / sample).read_text(encoding="utf-8").splitlines()
    after = apply_rewrites("\n".join(before), sample, rules).splitlines()
    before_set, after_set = set(before), set(after)
    removed = [ln for ln in before if ln not in after_set][:8]
    print(f"Exemple : {sample}")
    print(f"  lignes avant : {len(before)}  |  après : {len(after)}")
    print("\n  Lignes retirées / neutralisées (extrait) :")
    for ln in removed:
        print(f"    - {ln.strip()[:100]}")

Exemple : README.md
  lignes avant : 361  |  après : 360

  Lignes retirées / neutralisées (extrait) :
    - ├── CLAUDE.md                   # Context for Claude Code (read first)


## 7. Récapitulatif

In [8]:
recap = pd.DataFrame([
    {"indicateur": "Fichiers conservés", "valeur": n_kept},
    {"indicateur": "Fichiers exclus", "valeur": n_excluded},
    {"indicateur": "Docs neutres déposées", "valeur": n_neutral},
    {"indicateur": "Patterns interdits actifs", "valeur": len(config["forbidden_patterns"])},
    {"indicateur": "Traces source hors zones connues (doit être 0)", "valeur": len(findings_src)},
    {"indicateur": "Traces brutes (toutes zones)", "valeur": len(findings_raw)},
    {"indicateur": "Traces dans l'export jetable (doit être 0)", "valeur": len(findings_export)},
])
verdict = "✅ PRÊT" if not findings_src and not findings_export else "❌ À CORRIGER"
print(f"Verdict release : {verdict}")
recap

Verdict release : ❌ À CORRIGER


,indicateur,valeur
0,Fichiers conservés,5303
1,Fichiers exclus,3049
2,Docs neutres déposées,2
3,Patterns interdits actifs,9
4,Traces source hors zones connues (doit être 0),166
5,Traces brutes (toutes zones),3216
6,Traces dans l'export jetable (doit être 0),4


**Conclusion.** Si les deux compteurs de traces (source hors zones connues et export jetable) sont
à 0, le pipeline `make gitlab-release` produirait un dépôt propre. La publication réelle reste une
action manuelle ultérieure (`git remote add gitlab …` puis push), hors périmètre de ce rapport.